# 02: Modern Machine Learning Forecasting: Lag Windows, Rolling Stats & GBDT

**Track 07: Statistical Time-Series Forecasting** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Transform time series into supervised tabular regression: Rolling window moving averages, EWMA exponential smoothing, expanding statistics, autoregressive lags, and LightGBM forecasting.


## 1. Feature Engineering on Time Horizons
Construct lag features: $y_{t-1}, y_{t-2}, y_{t-7}$, rolling means, and rolling standard deviations.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import lightgbm as lgb

df = load_dataset("stock_market")
date_col = "Date" if "Date" in df.columns else df.columns[0]
close_col = "Close" if "Close" in df.columns else ("AAPL.Close" if "AAPL.Close" in df.columns else df.columns[1])

df["Date"] = pd.to_datetime(df[date_col])
df["Close"] = pd.to_numeric(df[close_col], errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)

# Autoregressive Lags
for lag in [1, 2, 3, 5, 10]:
    df[f"Close_Lag_{lag}"] = df["Close"].shift(lag)

# Rolling Windows
df["Rolling_Mean_7"] = df["Close"].shift(1).rolling(window=7).mean()
df["Rolling_Std_7"] = df["Close"].shift(1).rolling(window=7).std()

df = df.dropna().reset_index(drop=True)
print(f"Generated Feature Matrix: {df.shape}")
print(df[["Date", "Close", "Close_Lag_1", "Rolling_Mean_7", "Rolling_Std_7"]].tail())


## 2. Chronological Walk-Forward Split & LightGBM Model
Prevent lookahead bias using temporal train/test split.

In [ ]:
features = [c for c in df.columns if "Lag" in c or "Rolling" in c]
target = "Close"

split_idx = int(len(df) * 0.8)
X_train, X_test = df[features].iloc[:split_idx], df[features].iloc[split_idx:]
y_train, y_test = df[target].iloc[:split_idx], df[target].iloc[split_idx:]

model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=== Walk-Forward ML Forecast Results ===")
print(f"Test RMSE : ${rmse:.2f}")
print(f"Test MAPE : {mape:.2f}%")